# Chicago taxi trips, 11–17 September 2023

A descriptive summary of one week of **reported** taxi trips: when people ride, where trips start, how they pay and tip, and which companies carry them. All figures describe trips reported to the City of Chicago, not total market volume.

## Summary

_(filled in after first run)_

## 1. Setup and data loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.width", 200)

# Chart style: one blue for single series, orange as the second series; light grey recessive grid
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8a8a85"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 10,
})
thousands = mtick.FuncFormatter(lambda v, _: f"{v:,.0f}")

In [ ]:
raw = pd.read_csv("trips.csv", parse_dates=["trip_start_timestamp", "trip_end_timestamp"])
print(f"{len(raw):,} rows x {raw.shape[1]} columns")
print("trip_id unique:", raw["trip_id"].is_unique, "| exact duplicate rows:", raw.duplicated().sum())
print("Start range:", raw["trip_start_timestamp"].min(), "->", raw["trip_start_timestamp"].max())
print("Distinct taxi_id:", raw["taxi_id"].nunique(), "| distinct raw company strings:", raw["company"].nunique())

## 2. Data quality and cleaning

### 2.1 Missing values

In [ ]:
miss = raw.isna().sum()
miss = miss[miss > 0].to_frame("missing")
miss["share_of_rows_%"] = miss["missing"] / len(raw) * 100
miss

Reading the table:
- **Census tracts and dropoff areas** are missing by design (privacy suppression, trips outside Chicago). Census tracts are not used; geography uses community areas.
- **`pickup_community_area` is missing** for trips that start outside Chicago. This is a finding for Question 2, not a defect, so those rows are kept.
- **`fare`, `tips`, `tolls`, `extras`, `trip_total` are missing together** on 324 rows (almost all `Cash`). Without them a trip cannot be valued, so these rows are excluded (rule R1 below).
- `trip_seconds` is missing on 33 rows and one row has no `trip_end_timestamp`. Neither field is needed for the four questions, so these rows are kept unless another rule removes them.

### 2.2 Implausible values and money that does not add up

In [ ]:
r = raw.copy()
r["calc_total"] = r[["fare", "tips", "tolls", "extras"]].sum(axis=1, min_count=4)
r["gap"] = (r["trip_total"] - r["calc_total"]).round(2)
r["mph"] = np.where(r["trip_seconds"] > 0, r["trip_miles"] / (r["trip_seconds"] / 3600), np.nan)

print("Zero values (all rows):")
print(f"  trip_miles == 0     : {(r.trip_miles == 0).sum():>6,}")
print(f"  trip_seconds == 0   : {(r.trip_seconds == 0).sum():>6,}")
print(f"  fare == 0           : {(r.fare == 0).sum():>6,}")
print(f"  zero miles AND zero seconds: {((r.trip_miles == 0) & (r.trip_seconds == 0)).sum():,}")
print()
print("Extremes (all rows):")
print(r[["trip_seconds", "trip_miles", "fare", "tolls", "extras", "trip_total"]].quantile([.5, .99, .999, 1]).T.rename(columns={.5: "median", .99: "p99", .999: "p99.9", 1: "max"}))
print()
print("Gap = trip_total - (fare + tips + tolls + extras), rows with all components present:")
print(r["gap"].value_counts().sort_index().to_frame("rows").T.to_string())

In [ ]:
g = r.dropna(subset=["gap"])
print("Gap by payment type (rows):")
print(pd.crosstab(g["payment_type"], g["gap"].map(lambda v: "0.00" if v == 0 else ("0.50" if v == 0.5 else "other"))))
print()
print("Gap of 0.50 vs a recorded tip > 0 (rows):")
print(pd.crosstab(g["gap"].eq(0.5).rename("gap == 0.50"), (g["tips"] > 0).rename("tip > 0")))

**What the gap means.** In 46,522 rows `trip_total` is exactly $0.50 above the sum of its parts. This is almost entirely card and mobile payments, and it appears when a tip was recorded. It looks like a flat charge for electronic payment that has no column of its own. A further 1,676 rows (mostly Credit Card, mostly $2.50 or $3.00, roughly $0.50 plus a $2 or $2.50 item) have larger unexplained gaps. In every case `trip_total` is **at least** the sum of its parts and never less, so nothing looks like lost money.

**Decision:** I treat `trip_total` as the amount charged and use it for all revenue figures. I do not exclude rows for a gap. The gap is small (see the sensitivity check below), and a $0.50 gap on half the data is a feature of how totals are built, not an error. Tip percentages use `tips` and `fare` only, so they are unaffected.

Revenue therefore **includes tips, tolls, extras and the unitemised amounts**.

### 2.3 Cleaning rules

In [ ]:
c = raw.copy()
c["gap"] = (c["trip_total"] - c[["fare", "tips", "tolls", "extras"]].sum(axis=1, min_count=4)).round(2)
mph = np.where(c["trip_seconds"] > 0, c["trip_miles"] / (c["trip_seconds"] / 3600), np.nan)

rules = {
    "R1 missing fare/trip_total (cannot be valued)":
        c["trip_total"].isna() | c["fare"].isna(),
    "R2 fare == 0 (no revenue recorded)":
        c["fare"].eq(0),
    "R3 trip_total > $1,000 (data-entry errors, e.g. $4,444 extras)":
        c["trip_total"] > 1000,
    "R4 impossible speed (miles > 0 in 0 seconds, or > 100 mph)":
        ((c["trip_miles"] > 0) & c["trip_seconds"].eq(0)) | (pd.Series(mph, index=c.index) > 100),
    "R5 fare >= $100 with < 1 mile and < 10 minutes (meter fault)":
        (c["fare"] >= 100) & (c["trip_miles"] < 1) & (c["trip_seconds"] < 600),
}
flags = pd.DataFrame(rules)
first_hit = pd.Series(0, index=flags.columns)
remaining = pd.Series(True, index=c.index)
for col in flags.columns:
    first_hit[col] = (flags[col] & remaining).sum()
    remaining &= ~flags[col]

waterfall = pd.DataFrame({
    "rows flagged (any overlap)": flags.sum(),
    "rows removed (not already removed above)": first_hit,
})
waterfall.loc["TOTAL removed"] = [flags.any(axis=1).sum(), first_hit.sum()]
waterfall

In [ ]:
clean = c.loc[~flags.any(axis=1)].copy()
removed = len(raw) - len(clean)
print(f"Raw trips: {len(raw):,} | removed: {removed:,} ({removed/len(raw):.2%}) | analysed: {len(clean):,}")

**Values I deliberately keep, and why**
- **Zero-mile and zero-second trips** (about 10% of rows have zero miles). Most have small fares such as $3.25, the base meter charge, so they look like very short or aborted hails that were still reported. They are reported trips with a pickup time and place, so they stay in the counts. They pull down average distance and average fare slightly.
- **Long or expensive trips that are consistent** (for example 195 miles, $471 fare, 5 hours) are plausible out-of-town journeys and are kept.
- **Gaps between `trip_total` and its parts** (section 2.2): kept, `trip_total` used as reported.

**Standardising company names.** The same company appears under two spellings: *Taxicab Insurance Agency Llc* / *Taxicab Insurance Agency, LLC*, and *Choice Taxi Association* / *Choice Taxi Association Inc*. I merge both pairs, because counting them separately would split one company's trips and taxis. Other similar names (such as *Taxi Affiliation Services Llc - Yell…*, a 75-trip entry) I leave alone, since I can't tell from the data alone whether they are the same business.

**One analysis set.** Every question below uses the same cleaned data (the rows left after R1–R5, with merged company names), so totals reconcile across sections.

In [ ]:
company_map = {
    "Taxicab Insurance Agency, LLC": "Taxicab Insurance Agency Llc",
    "Choice Taxi Association Inc": "Choice Taxi Association",
}
n_merged = clean["company"].isin(company_map).sum()
clean["company"] = clean["company"].replace(company_map)
print(f"Rows with a renamed company: {n_merged:,} | distinct companies: {raw.company.nunique()} -> {clean.company.nunique()}")

# Time fields
clean["start"] = clean["trip_start_timestamp"]
clean["dow"] = clean["start"].dt.day_name()
clean["hour"] = clean["start"].dt.hour
clean["is_weekend"] = clean["start"].dt.dayofweek >= 5
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
clean["dow"] = pd.Categorical(clean["dow"], categories=days, ordered=True)

# Sensitivity: how much do the gap rows matter for revenue?
gap_total = clean["gap"].clip(lower=0).sum()
print(f"Total revenue (trip_total): ${clean.trip_total.sum():,.0f} | sum of unitemised gaps: ${gap_total:,.0f} ({gap_total/clean.trip_total.sum():.2%})")

**Assumptions at a glance**
1. Timestamps are local Chicago time (no time zone is given) and are rounded to the nearest 15 minutes. "Hour of trip start" is the hour of the rounded timestamp, so a trip that really began at 17:55 may be in the 18:00 hour. Durations use `trip_seconds`.
2. The week is Monday 11 to Sunday 17 September 2023, one of each weekday, so a weekday-versus-weekend comparison uses *average trips per day* (5 weekdays vs 2 weekend days).
3. "Revenue" is the sum of `trip_total`, which includes tips.
4. Cash tips are generally not recorded, so tipping is not compared like for like across payment types.
5. Community areas are shown by their number. Names would need outside data, which is out of scope.
6. This is a single week, so one-off events could affect individual days.

## 3. When is demand highest?

In [ ]:
by_day = clean.groupby("dow", observed=True).size().rename("trips").to_frame()
by_day["share_%"] = by_day["trips"] / by_day["trips"].sum() * 100
by_day["date"] = [d.strftime("%d %b") for d in sorted(clean["start"].dt.normalize().unique())]
by_day

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
cols = [ORANGE if d in ("Saturday", "Sunday") else BLUE for d in by_day.index]
bars = ax.bar(by_day.index, by_day["trips"], color=cols, width=0.65)
ax.bar_label(bars, labels=[f"{v:,.0f}" for v in by_day["trips"]], padding=3, fontsize=8, color="#52514e")
ax.set_title("Reported taxi trips by day of week, 11-17 Sep 2023")
ax.set_xlabel("Day of week (orange = weekend)")
ax.set_ylabel("Trips")
ax.yaxis.set_major_formatter(thousands)
ax.set_ylim(0, by_day["trips"].max() * 1.12)
plt.tight_layout(); plt.show()

_(takeaway)_

In [ ]:
by_hour = clean.groupby("hour").size().rename("trips").to_frame()
by_hour["share_%"] = by_hour["trips"] / by_hour["trips"].sum() * 100
print("Busiest 3 hours:"); print(by_hour.nlargest(3, "trips"))
print("\nQuietest 3 hours:"); print(by_hour.nsmallest(3, "trips"))

# average trips per day for each hour, weekday vs weekend
n_days = clean.groupby("is_weekend")["start"].apply(lambda s: s.dt.normalize().nunique())
prof = (clean.groupby(["is_weekend", "hour"]).size().unstack(0)
        .rename(columns={False: "Weekday (Mon-Fri)", True: "Weekend (Sat-Sun)"}))
prof["Weekday (Mon-Fri)"] /= n_days[False]
prof["Weekend (Sat-Sun)"] /= n_days[True]
prof = prof.reindex(range(24), fill_value=0)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(prof.index, prof["Weekday (Mon-Fri)"], color=BLUE, lw=2, marker="o", ms=4, label="Weekday (Mon-Fri)")
ax.plot(prof.index, prof["Weekend (Sat-Sun)"], color=ORANGE, lw=2, marker="o", ms=4, label="Weekend (Sat-Sun)")
ax.set_title("Average trips per day by hour of trip start: weekdays vs weekend")
ax.set_xlabel("Hour of trip start (0 = midnight, local time)")
ax.set_ylabel("Average trips per day")
ax.set_xticks(range(0, 24, 2))
ax.yaxis.set_major_formatter(thousands)
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
heat = clean.groupby(["dow", "hour"], observed=True).size().unstack("hour").reindex(columns=range(24), fill_value=0)
fig, ax = plt.subplots(figsize=(10, 3.8))
im = ax.imshow(heat.values, aspect="auto", cmap="Blues")
ax.set_xticks(range(0, 24, 2)); ax.set_xticklabels(range(0, 24, 2))
ax.set_yticks(range(7)); ax.set_yticklabels(days)
ax.grid(False)
ax.set_title("Trips by day of week and hour of trip start")
ax.set_xlabel("Hour of trip start (0 = midnight, local time)")
ax.set_ylabel("Day of week")
cb = fig.colorbar(im, ax=ax, pad=0.01); cb.set_label("Trips")
plt.tight_layout(); plt.show()

stack = heat.stack()
print("Busiest day-hour:", stack.idxmax(), f"({stack.max():,} trips)")
print("Quietest day-hour:", stack.idxmin(), f"({stack.min():,} trips)")
print("\nWeekday vs weekend:")
wk = pd.DataFrame({"avg trips per day": clean.groupby("is_weekend").size() / n_days}).rename(index={False: "Weekday", True: "Weekend"})
wk

_(takeaway)_

## 4. Where do trips start?

In [ ]:
pu = clean.groupby("pickup_community_area", dropna=False).agg(trips=("trip_id", "size"), revenue=("trip_total", "sum"))
no_area = clean["pickup_community_area"].isna()
print(f"Trips with no pickup community area: {no_area.sum():,} = {no_area.mean():.2%} of analysed trips"
      f" (raw data: {raw.pickup_community_area.isna().sum():,} = {raw.pickup_community_area.isna().mean():.2%})")
print(f"  ...and {clean.loc[no_area,'trip_total'].sum()/clean.trip_total.sum():.2%} of revenue")
pu = pu.drop(index=np.nan, errors="ignore")
pu.index = pu.index.astype(int)
pu["trip_share_%"] = pu["trips"] / len(clean) * 100
pu["revenue_share_%"] = pu["revenue"] / clean["trip_total"].sum() * 100
top_n = pu.nlargest(10, "trips"); top_r = pu.nlargest(10, "revenue")
print()
print("Top 10 by trip count"); display(top_n.assign(rank_revenue=pu["revenue"].rank(ascending=False).astype(int)[top_n.index]))
print("Top 10 by total trip_total"); display(top_r.assign(rank_trips=pu["trips"].rank(ascending=False).astype(int)[top_r.index]))
print("Overlap of the two top-10 lists:", len(set(top_n.index) & set(top_r.index)), "areas |",
      f"top 10 by trips hold {top_n.trips.sum()/len(clean):.1%} of all trips")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
for ax, d, col, title, xl in [
    (axes[0], top_n, "trips", "Top 10 pickup areas by trips", "Trips"),
    (axes[1], top_r, "revenue", "Top 10 pickup areas by total revenue", "Total trip_total ($)")]:
    d = d.sort_values(col)
    ax.barh(d.index.astype(str), d[col], color=BLUE, height=0.65)
    ax.set_title(title); ax.set_xlabel(xl); ax.set_ylabel("Pickup community area (number)")
    ax.xaxis.set_major_formatter(thousands); ax.grid(axis="y", visible=False)
fig.suptitle("Where trips start, 11-17 Sep 2023 (trips with a Chicago pickup area)", y=1.01, fontsize=11)
plt.tight_layout(); plt.show()

_(takeaway)_

## 5. How do riders pay and tip?

In [ ]:
pay = clean.groupby("payment_type").agg(
    trips=("trip_id", "size"),
    revenue=("trip_total", "sum"),
    avg_fare=("fare", "mean"),
    total_fare=("fare", "sum"),
    total_tips=("tips", "sum"),
    trips_with_tip=("tips", lambda s: (s > 0).sum()),
).sort_values("trips", ascending=False)
pay["trip_share_%"] = pay["trips"] / pay["trips"].sum() * 100
pay["revenue_share_%"] = pay["revenue"] / pay["revenue"].sum() * 100
pay["tips_%_of_fare"] = pay["total_tips"] / pay["total_fare"] * 100
pay["trips_with_tip_%"] = pay["trips_with_tip"] / pay["trips"] * 100
pay["median_tip_%_(tipped trips)"] = clean[clean.tips > 0].assign(p=lambda d: d.tips / d.fare * 100).groupby("payment_type")["p"].median()
pay_table = pay[["trips", "trip_share_%", "revenue", "revenue_share_%", "avg_fare", "tips_%_of_fare", "trips_with_tip_%", "median_tip_%_(tipped trips)"]]
pay_table.style.format({"trips": "{:,.0f}", "revenue": "${:,.0f}", "avg_fare": "${:.2f}", "trip_share_%": "{:.1f}", "revenue_share_%": "{:.1f}",
                        "tips_%_of_fare": "{:.1f}", "trips_with_tip_%": "{:.1f}", "median_tip_%_(tipped trips)": "{:.1f}"}, na_rep="n/a")

In [ ]:
card = clean[clean.payment_type.isin(["Credit Card", "Mobile"])]
print(f"Credit Card + Mobile only: tips = {card.tips.sum()/card.fare.sum():.1%} of fare "
      f"({(card.tips > 0).mean():.1%} of these trips have a tip)")
print(f"All payment types pooled : tips = {clean.tips.sum()/clean.fare.sum():.1%} of fare (understated, cash tips are not recorded)")

In [ ]:
order = pay.index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
axes[0].bar(order, pay["trips"], color=BLUE, width=0.65)
axes[0].set_title("Trips by payment type"); axes[0].set_ylabel("Trips"); axes[0].yaxis.set_major_formatter(thousands)
axes[1].bar(order, pay["revenue"], color=BLUE, width=0.65)
axes[1].set_title("Revenue by payment type"); axes[1].set_ylabel("Total trip_total ($)"); axes[1].yaxis.set_major_formatter(thousands)
card_pt = pay.loc[["Credit Card", "Mobile"]]
b = axes[2].bar(card_pt.index, card_pt["tips_%_of_fare"], color=BLUE, width=0.5)
axes[2].bar_label(b, labels=[f"{v:.1f}%" for v in card_pt["tips_%_of_fare"]], padding=3, fontsize=9, color="#52514e")
axes[2].set_title("Tips as % of fare (card and mobile only)"); axes[2].set_ylabel("Tips as % of total fare")
axes[2].set_ylim(0, card_pt["tips_%_of_fare"].max() * 1.2)
for ax in axes: ax.set_xlabel("Payment type")
for ax in axes[:2]: ax.tick_params(axis="x", rotation=40)
plt.tight_layout(); plt.show()

_(takeaway)_

## 6. Which companies carry the trips?

In [ ]:
co = clean.groupby("company").agg(trips=("trip_id", "size"), revenue=("trip_total", "sum"), distinct_taxis=("taxi_id", "nunique"))
co["share_of_all_trips_%"] = co["trips"] / len(clean) * 100
co["trips_per_taxi"] = co["trips"] / co["distinct_taxis"]
co["revenue_per_taxi"] = co["revenue"] / co["distinct_taxis"]
top_co = co.nlargest(10, "trips")
display(top_co.style.format({"trips": "{:,.0f}", "revenue": "${:,.0f}", "distinct_taxis": "{:,.0f}", "share_of_all_trips_%": "{:.1f}",
                             "trips_per_taxi": "{:.1f}", "revenue_per_taxi": "${:,.0f}"}))
print(f"Top 10 companies: {top_co.trips.sum()/len(clean):.1%} of analysed trips, {top_co.revenue.sum()/clean.trip_total.sum():.1%} of revenue")
print(f"Distinct companies in total: {co.shape[0]} | distinct taxis in total: {clean.taxi_id.nunique():,}")
multi = clean.groupby("taxi_id")["company"].nunique()
print(f"Taxis seen under more than one company: {(multi > 1).sum()} (so company taxi counts can overlap slightly)")

In [ ]:
d = top_co.sort_values("trips")
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.barh(d.index, d["trips"], color=BLUE, height=0.65)
for y, (t, s) in enumerate(zip(d["trips"], d["share_of_all_trips_%"])):
    ax.text(t, y, f"  {t:,.0f} ({s:.1f}%)", va="center", fontsize=8, color="#52514e")
ax.set_title("Top 10 taxi companies by reported trips, 11-17 Sep 2023")
ax.set_xlabel("Trips (share of all analysed trips in brackets)"); ax.set_ylabel("Company")
ax.xaxis.set_major_formatter(thousands); ax.grid(axis="y", visible=False)
ax.set_xlim(0, d["trips"].max() * 1.22)
plt.tight_layout(); plt.show()

_(takeaway)_

## 7. Possible follow-up

_(follow-up)_